# Unidade 2 · Aprendizagem supervisionada
**Aprendizagem Computacional · 4º TIA · Apostila do Aluno · versão INICIAL**

Este notebook cobre os cap. 5 e 8 a 16 da apostila em duas partes, sempre com o **mesmo protocolo**:
**separar → baseline → modelos dentro de Pipeline → validação cruzada só no treino → escolher → teste uma única vez → interpretar**.

- **Parte A · Regressão:** progressão do diabetes um ano após a consulta (`load_diabetes`, 442 pacientes).
- **Parte B · Classificação:** tumor maligno ou benigno a partir de medidas de imagem (`load_breast_cancer`, 569 casos).

Os dados vêm do próprio scikit-learn, então não é preciso internet. `random_state=42` em tudo que sorteia.

**Como usar a versão INICIAL:** procure os `# TODO`. Cada um corresponde a um capítulo da apostila.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, KFold, StratifiedKFold, cross_validate, GridSearchCV
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import StandardScaler
import warnings
from sklearn.exceptions import UndefinedMetricWarning
warnings.filterwarnings('ignore', category=UndefinedMetricWarning)  # o baseline nunca prevê 'maligno'
SEMENTE = 42
pd.set_option('display.precision', 3)

# Parte A · Regressão (cap. 5, 8 e 10)
## A1 · Dados, split e baseline

In [ ]:
from sklearn.datasets import load_diabetes
diab = load_diabetes(as_frame=True)
Xr, yr = diab.data, diab.target
print(Xr.shape); print(yr.describe().round(1))
Xr_tr, Xr_te, yr_tr, yr_te = train_test_split(Xr, yr, test_size=0.2, random_state=SEMENTE)
cv_r = KFold(n_splits=5, shuffle=True, random_state=SEMENTE)

As 10 colunas (idade, sexo, IMC, pressão e 6 medidas do sangue) já vêm centradas e escaladas pelo próprio scikit-learn. Mesmo assim, mantemos o `StandardScaler` no pipeline por hábito: na base do seu projeto, isso não virá pronto.

In [ ]:
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression, Ridge, LassoCV

def cv_regressao(modelo):
    r = cross_validate(modelo, Xr_tr, yr_tr, cv=cv_r,
                       scoring=['neg_mean_absolute_error', 'neg_root_mean_squared_error', 'r2'])
    return {'MAE': -r['test_neg_mean_absolute_error'].mean(),
            'RMSE': -r['test_neg_root_mean_squared_error'].mean(),
            'R2': r['test_r2'].mean()}

regressores = {
    'Baseline (média)': None,   # TODO: DummyRegressor (cap. 5)
    'Linear': make_pipeline(StandardScaler(), LinearRegression()),
    'Ridge (alpha=10)': make_pipeline(StandardScaler(), Ridge(alpha=10)),
    'Lasso (alpha por CV)': make_pipeline(StandardScaler(), LassoCV(cv=5, random_state=SEMENTE)),
}
tabela_r = pd.DataFrame({n: cv_regressao(m) for n, m in regressores.items()}).T
tabela_r

**Leitura:** os três modelos lineares ficam muito parecidos e todos superam bem o baseline. O R² perto de 0,5 diz que metade da variação da doença é explicada pelas 10 medidas; a outra metade depende de coisas que não estão na base.

**Pergunta para discussão:** por que o RMSE é sempre maior que o MAE?

**Resposta esperada:** o RMSE eleva os erros ao quadrado antes de tirar a média, então erros grandes pesam mais. Se o RMSE for muito maior que o MAE, há alguns erros grandes; olhe os resíduos.

In [ ]:
lasso = regressores['Lasso (alpha por CV)'].fit(Xr_tr, yr_tr)
coefs = pd.Series(lasso[-1].coef_, index=Xr.columns)
print('alpha escolhido pelo LassoCV:', round(lasso[-1].alpha_, 3))
print('Coeficientes zerados pelo Lasso:', list(coefs[coefs == 0].index))
coefs.sort_values().plot.barh(figsize=(6, 3.5), title='Coeficientes do Lasso (dados padronizados)')
plt.axvline(0, color='grey', lw=0.8); plt.tight_layout(); plt.show()

**Regularização na prática (cap. 8 e 10):** o Lasso penaliza a soma dos valores absolutos dos coeficientes e, por isso, consegue **zerar** os menos úteis. É seleção de variáveis automática e um exemplo concreto da ideia teórica do cap. 10: restringir a complexidade do modelo para generalizar melhor.

In [ ]:
from sklearn.metrics import mean_absolute_error, r2_score
final_r = regressores['Linear'].fit(Xr_tr, yr_tr)
pred = final_r.predict(Xr_te)
print(f'TESTE · Linear: MAE = {mean_absolute_error(yr_te, pred):.1f} | R2 = {r2_score(yr_te, pred):.3f}')
print(f'TESTE · Baseline: MAE = {mean_absolute_error(yr_te, np.full_like(pred, yr_tr.mean())):.1f}')
plt.scatter(pred, yr_te - pred, s=14); plt.axhline(0, color='grey')
plt.xlabel('Previsto'); plt.ylabel('Resíduo (real − previsto)'); plt.title('Resíduos no teste'); plt.show()

**Leitura dos resíduos:** se os pontos formam uma nuvem sem padrão em torno do zero, o modelo linear capturou o que podia. Um funil (resíduos crescendo com o previsto) ou uma curva indicariam que falta algo (transformar y, incluir interações ou usar um modelo não linear).

# Parte B · Classificação (cap. 9 e 11 a 16)
## B1 · Dados e o que é \"positivo\"
No scikit-learn, `target = 0` é maligno. Recodificamos para que **1 = maligno**, a classe que não queremos deixar passar. As métricas de precisão e recall sempre se referem à classe positiva.

In [ ]:
from sklearn.datasets import load_breast_cancer
bc = load_breast_cancer(as_frame=True)
Xc = bc.data
yc = (bc.target == 0).astype(int)   # 1 = maligno
print(Xc.shape, '| proporção de malignos:', round(yc.mean(), 3))
# TODO: split 75/25 ESTRATIFICADO por yc, random_state=SEMENTE (cap. 5)
Xc_tr, Xc_te, yc_tr, yc_te = ...
cv_c = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEMENTE)

## B2 · Regressão logística e suas métricas (cap. 9)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import classification_report, ConfusionMatrixDisplay

logit = make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000)).fit(Xc_tr, yc_tr)
dummy = DummyClassifier(strategy='most_frequent').fit(Xc_tr, yc_tr)
print('Baseline (sempre benigno) · acurácia no treino-CV:',
      cross_validate(dummy, Xc_tr, yc_tr, cv=cv_c)['test_score'].mean().round(3))
from sklearn.model_selection import cross_val_predict
pred_cv = cross_val_predict(logit, Xc_tr, yc_tr, cv=cv_c)
print(classification_report(yc_tr, pred_cv, target_names=['benigno', 'maligno'], digits=3))
ConfusionMatrixDisplay.from_predictions(yc_tr, pred_cv, display_labels=['benigno', 'maligno'], cmap='Blues')
plt.title('Logística · previsões fora da dobra (treino)'); plt.show()

**Leitura:** o baseline acerta cerca de 63% sem olhar nada, porque 63% dos casos são benignos. Ele tem recall zero para maligno, ou seja, é inútil. A logística erra poucos casos, e a matriz mostra **quais** erros: falsos negativos (maligno dito benigno) são os mais graves aqui.

In [ ]:
from sklearn.metrics import precision_recall_curve, roc_auc_score, average_precision_score
proba_cv = cross_val_predict(logit, Xc_tr, yc_tr, cv=cv_c, method='predict_proba')[:, 1]
print('ROC-AUC:', round(roc_auc_score(yc_tr, proba_cv), 3), '| PR-AUC (average precision):',
      round(average_precision_score(yc_tr, proba_cv), 3))
linhas = []
for limiar in [0.5, 0.4, 0.3, 0.2, 0.1]:
    p = (proba_cv >= limiar).astype(int)
    vp = ((p == 1) & (yc_tr == 1)).sum(); fn = ((p == 0) & (yc_tr == 1)).sum(); fp = ((p == 1) & (yc_tr == 0)).sum()
    linhas.append({'limiar': limiar, 'recall': vp / (vp + fn), 'precisão': vp / (vp + fp),
                   'malignos perdidos (FN)': fn, 'alarmes falsos (FP)': fp})
pd.DataFrame(linhas)

**O limiar é uma decisão de negócio (cap. 9).** Baixar o limiar de 0,5 para 0,3 troca alguns alarmes falsos por menos malignos perdidos. Quem escolhe o limiar é o custo de cada erro no problema real, não o algoritmo.

**Pergunta para discussão:** num programa público de triagem, que limiar você defenderia, e o que precisa saber para decidir?

**Resposta esperada:** um limiar mais baixo, porque perder um maligno custa muito mais que um exame extra. Para decidir, é preciso saber a capacidade de exames confirmatórios (quantos alarmes o serviço aguenta) e o custo de cada tipo de erro.

## B3 · Teoria na prática: complexidade × generalização (cap. 10 e 13)
Uma árvore de decisão com profundidade crescente: o desempenho no treino sobe sempre, mas na validação ele para de subir. A distância entre as curvas é o **sobreajuste**.

In [ ]:
from sklearn.tree import DecisionTreeClassifier
prof = range(1, 13)
treino, valid = [], []
for d in prof:
    r = cross_validate(DecisionTreeClassifier(max_depth=d, random_state=SEMENTE), Xc_tr, yc_tr,
                       cv=cv_c, scoring='f1', return_train_score=True)
    treino.append(r['train_score'].mean()); valid.append(r['test_score'].mean())
plt.plot(prof, treino, marker='o', label='treino'); plt.plot(prof, valid, marker='o', label='validação (CV)')
plt.xlabel('max_depth'); plt.ylabel('F1 (maligno)'); plt.legend(); plt.grid(alpha=0.3)
plt.title('Curva de complexidade da árvore'); plt.show()
print('Melhor profundidade na validação:', list(prof)[int(np.argmax(valid))])

## B4 · Comparação justa de famílias de modelos (cap. 11 a 16)
Mesmas 5 dobras estratificadas, mesmas métricas, hiperparâmetros escolhidos por `GridSearchCV` **dentro** do treino. A métrica principal é o F1 da classe maligna, e acompanhamos o recall.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier

def busca(modelo, grade):
    return GridSearchCV(modelo, grade, cv=cv_c, scoring='f1')

modelos = {
    'Baseline': DummyClassifier(strategy='most_frequent'),
    'Logística (cap. 9)': make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000)),
    # TODO (cap. 11): k-NN com StandardScaler, buscando n_neighbors em [3, 5, 7, 11, 15]
    'k-NN (cap. 11)': None,
    'Naive Bayes (cap. 12)': make_pipeline(StandardScaler(), GaussianNB()),
    'MLP (cap. 15)': make_pipeline(StandardScaler(), MLPClassifier(hidden_layer_sizes=(32,), alpha=1e-3,
                                                               max_iter=2000, random_state=SEMENTE)),
    'Árvore (cap. 13)': busca(DecisionTreeClassifier(random_state=SEMENTE), {'max_depth': [2, 3, 4, 5, 6, None]}),
    'SVM RBF (cap. 14)': busca(Pipeline([('esc', StandardScaler()), ('m', SVC(random_state=SEMENTE))]),
                               {'m__C': [0.1, 1, 10], 'm__gamma': ['scale', 0.01]}),
    'Random Forest (cap. 16)': RandomForestClassifier(n_estimators=300, random_state=SEMENTE),
    'Gradient Boosting (cap. 16)': GradientBoostingClassifier(random_state=SEMENTE),
}
modelos['Votação suave (cap. 16)'] = VotingClassifier(
    [('lr', modelos['Logística (cap. 9)']), ('mlp', modelos['MLP (cap. 15)']),
     ('rf', modelos['Random Forest (cap. 16)'])], voting='soft')

linhas = []
for nome, m in modelos.items():
    r = cross_validate(m, Xc_tr, yc_tr, cv=cv_c, scoring=['f1', 'recall', 'precision', 'roc_auc'])
    linhas.append({'modelo': nome, 'F1': r['test_f1'].mean(), 'desvio F1': r['test_f1'].std(),
                   'recall': r['test_recall'].mean(), 'precisão': r['test_precision'].mean(),
                   'ROC-AUC': r['test_roc_auc'].mean()})
comparacao = pd.DataFrame(linhas).sort_values('F1', ascending=False).reset_index(drop=True)
comparacao

**Como ler a tabela:** quando as diferenças de F1 entre os primeiros colocados são menores que o desvio entre dobras, eles estão **empatados** na prática. Nesse caso, desempate por simplicidade e interpretabilidade, e não pela terceira casa decimal. Note também que o baseline tem F1 zero: ele nunca aponta um maligno.

**Pergunta para discussão:** a árvore isolada fica abaixo da Random Forest. Por quê?

**Resposta esperada:** uma árvore sozinha tem variância alta (muda muito com pequenas mudanças nos dados). A Random Forest faz a média de centenas de árvores treinadas em amostras e colunas sorteadas, o que reduz essa variância (cap. 16).

## B5 · Escolher e abrir o teste UMA vez
Escolhemos a **regressão logística**: ela fica entre as melhores na validação cruzada, é simples, gera probabilidades bem comportadas e tem coeficientes interpretáveis. Só agora usamos o teste.

In [ ]:
from sklearn.metrics import f1_score, recall_score, precision_score
final_c = modelos['Logística (cap. 9)'].fit(Xc_tr, yc_tr)
pred_te = final_c.predict(Xc_te)
print(f'TESTE · F1 = {f1_score(yc_te, pred_te):.3f} | recall = {recall_score(yc_te, pred_te):.3f} | '
      f'precisão = {precision_score(yc_te, pred_te):.3f}')
ConfusionMatrixDisplay.from_predictions(yc_te, pred_te, display_labels=['benigno', 'maligno'], cmap='Blues')
plt.title('Logística · teste (usado uma vez)'); plt.show()

## Checklist da Unidade 2
- [ ] Defini a classe positiva e escolhi a métrica pelo custo dos erros.
- [ ] Tenho baseline e ele aparece na tabela.
- [ ] Usei split estratificado e validação cruzada estratificada (classificação).
- [ ] Hiperparâmetros escolhidos por `GridSearchCV` só no treino.
- [ ] Comparei famílias de modelos com o mesmo protocolo e li o desvio entre dobras.
- [ ] Abri o teste uma única vez, depois da escolha.
- [ ] Discuti o limiar de decisão.

**No seu projeto (entregas de 17/11 e 24/11):** a seção A ou B deste notebook é o molde da sua tabela \"baseline × modelos\". Troque os dados, mantenha o protocolo.